# Análise das amostras — SpectralCrops

Lê o pacote exportado pelo SpectralCrops (**Annotations → ⬇ Exportar → Classificação**) e monta um
DataFrame com **uma linha por amostra**: label, bandeja, contornos e a **reflectância média em cada
uma das 8 bandas**, calculada dentro do contorno da amostra.

**Estrutura esperada** (a pasta extraída do `.zip`, onde este notebook já vem):

```
amostras_<data>/
  anotacoes.json             índice único: projetos, imagens, caminhos, observações e amostras
  analise_amostras.ipynb     <- este notebook
  LEIA-ME.txt
  <projeto>/<imagem>/
    365nm.png ... 850nm.png  bandas calibradas (reflectância = pixel / 255)
    rgb.png
```

**Seções:** 1 · Imports — 2 · Configuração — 3 · Funções — 4 · DataFrame — 5 · Conferência visual —
6 · Treino de modelos (a fazer)

## 1 - IMPORTS

In [ ]:
import os
import json

import numpy as np
import pandas as pd
import cv2 as cv
import matplotlib.pyplot as plt

## 2 - CONFIGURAÇÃO

- `PASTA_EXPORT`: pasta extraída do `.zip` (onde este notebook está, por padrão).
- `CONTORNO_MASCARA`: qual contorno da amostra vira a máscara, em ordem de preferência. Uma amostra
  pode ter vários (ex.: polígono + box + ponto da mesma semente).
- `EROSAO_PX`: encolhe a máscara alguns pixels antes de medir. As imagens atuais têm a borda das
  sementes borrada (~10–19 px), então a borda mistura semente com fundo; erodir mede só o interior.
  `0` = mede o contorno inteiro, como o "Calcular ROIs" do app.
- `RAIO_PONTO_PX`: janela em volta de um contorno do tipo ponto (mesma do app: ±8 px).

In [ ]:
PASTA_EXPORT = "."
CONTORNO_MASCARA = ["polygon", "ellipse", "box", "point"]
EROSAO_PX = 0
RAIO_PONTO_PX = 8

with open(os.path.join(PASTA_EXPORT, "anotacoes.json"), encoding="utf-8") as f:
    INDICE = json.load(f)
PROJETOS = INDICE["projetos"]          # {id: {nome, modo_calibracao, observacao_parametrizacao}}
IMAGENS = INDICE["imagens"]
print(f"{len(IMAGENS)} imagem(ns) de {len(PROJETOS)} projeto(s), exportado em {INDICE['exportado_em']}")
for pid, p in PROJETOS.items():
    print(f"  projeto {pid}: {p['nome']} (calibração: {p['modo_calibracao']})")

## 3 - FUNÇÕES

In [ ]:
def carregar_bandas(img):
    """(altura, largura, n_bandas) em reflectância 0–1 de uma imagem do índice. Banda
    sem calibração (bandas_cruas) vira NaN -- não é reflectância."""
    escala = INDICE.get("reflectancia_escala", 255)
    bandas = []
    for nm in INDICE["bandas_nm"]:
        caminho = img["bandas"].get(str(nm))
        if caminho and nm in img["bandas_calibradas"]:
            banda = cv.imread(os.path.join(PASTA_EXPORT, caminho), cv.IMREAD_UNCHANGED).astype(np.float32) / escala
        else:
            banda = np.full((img["altura"], img["largura"]), np.nan, np.float32)
        bandas.append(banda)
    return np.stack(bandas, axis=-1)


def escolher_contorno(amostra):
    """O contorno usado como máscara, seguindo CONTORNO_MASCARA."""
    por_tipo = {}
    for c in amostra["contornos"]:
        por_tipo.setdefault(c["tipo"], c)
    for tipo in CONTORNO_MASCARA:
        if tipo in por_tipo:
            return por_tipo[tipo]
    return None


def mascara(contorno, altura, largura, erosao_px=0):
    """Máscara booleana (altura, largura) do contorno."""
    m = np.zeros((altura, largura), np.uint8)
    if contorno["tipo"] in ("polygon", "ellipse"):
        cv.fillPoly(m, [np.array(contorno["pontos"], np.int32)], 1)
    elif contorno["tipo"] == "box":
        x1, y1, x2, y2 = contorno["caixa"]
        m[min(y1, y2):max(y1, y2), min(x1, x2):max(x1, x2)] = 1
    elif contorno["tipo"] == "point":
        x, y = contorno["ponto"]
        m[max(0, y - RAIO_PONTO_PX):y + RAIO_PONTO_PX, max(0, x - RAIO_PONTO_PX):x + RAIO_PONTO_PX] = 1
    if erosao_px > 0 and contorno["tipo"] != "point":
        k = cv.getStructuringElement(cv.MORPH_ELLIPSE, (2 * erosao_px + 1, 2 * erosao_px + 1))
        m = cv.erode(m, k)
    return m.astype(bool)


def recorte_amostra(cubo, m, margem_px=10):
    """Recorte (h, w, n_bandas) em volta da amostra, com o fundo zerado -- pra modelos de imagem (CNN)."""
    ys, xs = np.where(m)
    y0, y1 = max(0, ys.min() - margem_px), min(m.shape[0], ys.max() + margem_px + 1)
    x0, x1 = max(0, xs.min() - margem_px), min(m.shape[1], xs.max() + margem_px + 1)
    rec = cubo[y0:y1, x0:x1].copy()
    rec[~m[y0:y1, x0:x1]] = 0
    return rec

## 4 - DATAFRAME

Uma linha por amostra. Colunas:

- **identificação:** `imagem`, `pasta`, `aquisicao_id` (bandeja), `capturada_em`, `amostra`
- **projeto:** `projeto_id`, `projeto`, `modo_calibracao` (`mean` = média da região, `pixel` = pixel a pixel)
- **imagem:** `rotulo_imagem`, `observacao`
- **label:** `label` e `label_origem` (`amostra` = categoria da própria forma; `imagem` = rótulo 🏷 da imagem)
- **contornos:** `contornos` (todos os tipos da amostra), `contorno_mascara` (o usado na máscara)
- **geometria:** `area_px` (pixels na máscara), `largura_px`, `altura_px`
- **espectro:** `refl_<nm>` = reflectância média (%) dentro da máscara, uma coluna por banda

In [ ]:
linhas = []
for img in IMAGENS:
    projeto = PROJETOS[str(img["projeto_id"])]
    cubo = carregar_bandas(img)
    altura, largura = cubo.shape[:2]
    for amostra in img["amostras"]:
        contorno = escolher_contorno(amostra)
        if contorno is None:
            continue
        m = mascara(contorno, altura, largura, EROSAO_PX)
        if not m.any():
            continue
        ys, xs = np.where(m)
        linha = {
            "imagem": img["imagem"], "pasta": img["pasta"], "aquisicao_id": img["aquisicao_id"],
            "capturada_em": img["capturada_em"], "projeto_id": img["projeto_id"], "projeto": projeto["nome"],
            "modo_calibracao": projeto["modo_calibracao"],
            "rotulo_imagem": img["rotulo_imagem"], "observacao": img["observacao"],
            "amostra": amostra["amostra"], "label": amostra["label"], "label_origem": amostra["label_origem"],
            "contornos": "+".join(c["tipo"] for c in amostra["contornos"]),
            "contorno_mascara": contorno["tipo"],
            "area_px": int(m.sum()), "largura_px": int(xs.max() - xs.min() + 1), "altura_px": int(ys.max() - ys.min() + 1),
        }
        medias = np.nanmean(cubo[m], axis=0) * 100
        for nm, v in zip(INDICE["bandas_nm"], medias):
            linha[f"refl_{nm}"] = round(float(v), 2)
        linhas.append(linha)

df = pd.DataFrame(linhas)
BANDAS = [c for c in df.columns if c.startswith("refl_")]
print(f"{len(df)} amostras, {df['aquisicao_id'].nunique()} bandejas, {df['label'].nunique()} labels")
df.head()

In [ ]:
# amostras por label e por bandeja -- confira se as classes estão certas antes de treinar
pd.crosstab([df["projeto"], df["imagem"]], df["label"], margins=True, margins_name="total")

In [ ]:
# salva o DataFrame (abre no Excel; ; como separador e BOM pra acentuação certa)
df.to_csv(os.path.join(PASTA_EXPORT, "amostras_dataframe.csv"), sep=";", index=False, encoding="utf-8-sig")

## 5 - CONFERÊNCIA VISUAL

In [ ]:
# uma imagem com os contornos usados como máscara e o número de cada amostra
img = IMAGENS[0]
rgb = cv.cvtColor(cv.imread(os.path.join(PASTA_EXPORT, img["rgb"])), cv.COLOR_BGR2RGB)
for amostra in img["amostras"]:
    contorno = escolher_contorno(amostra)
    if contorno is None:
        continue
    m = mascara(contorno, rgb.shape[0], rgb.shape[1], EROSAO_PX).astype(np.uint8)
    cs, _ = cv.findContours(m, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    cv.drawContours(rgb, cs, -1, (0, 255, 0), 2)
    if cs:
        x, y, w, h = cv.boundingRect(cs[0])
        cv.putText(rgb, str(amostra["amostra"]), (x + w // 2 - 8, y + h // 2 + 6), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
plt.figure(figsize=(12, 7))
plt.imshow(rgb)
plt.title(f"{img['imagem']} ({PROJETOS[str(img['projeto_id'])]['nome']}) — {len(img['amostras'])} amostras")
plt.axis("off")
plt.show()

In [ ]:
# espectro médio por label (linha) ± desvio padrão (faixa)
nms = [int(c.split("_")[1]) for c in BANDAS]
plt.figure(figsize=(10, 5))
for label, grupo in df.groupby("label"):
    media, desvio = grupo[BANDAS].mean().values, grupo[BANDAS].std().values
    plt.plot(nms, media, marker="o", linewidth=2, label=f"{label} (n={len(grupo)})")
    plt.fill_between(nms, media - desvio, media + desvio, alpha=0.15)
plt.xlabel("Comprimento de onda (nm)")
plt.ylabel("Reflectância média (%)")
plt.ylim(0, 105)
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("Espectro médio por label")
plt.show()

## 6 - TREINO DE MODELOS (a fazer)

Próximas células: classificação pelo espectro médio (`df[BANDAS]` → `label`) e por imagem
(`recorte_amostra`). Validação **por bandeja** (`aquisicao_id`) -- ex.: `GroupKFold` do scikit-learn --,
nunca por amostra: sementes da mesma bandeja compartilham luz, lote e posição, e cair dos dois lados
infla a acurácia.

In [ ]:
# exemplo de recorte de uma amostra (8 bandas, fundo zerado) -- entrada pra uma CNN
img = IMAGENS[0]
cubo = carregar_bandas(img)
contorno = escolher_contorno(img["amostras"][0])
rec = recorte_amostra(cubo, mascara(contorno, *cubo.shape[:2], EROSAO_PX))
print("recorte:", rec.shape)